In [2]:
%%pyspark
# Fabric notebook source
# 02_Silver_Daily
# Incremental, rerun-safe Silver layer with SCD Type 2 history preservation.
# IMPORTANT: This notebook does NOT drop LH_Silver.

required = ["LH_Silver.driver_scd","LH_Silver.passenger_scd","LH_Silver.city_scd","LH_Silver.vehicle_type_scd","LH_Silver.map_vehicle_makes","LH_Silver.map_payment_methods","LH_Silver.map_ride_statuses","LH_Silver.map_cancellation_reasons","LH_Silver.rides_obt"]
missing = [t for t in required if not spark.catalog.tableExists(t)]
if missing: raise RuntimeError(f"Silver setup missing tables: {missing}")
print("Silver prerequisites validated.")

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 4, Finished, Available, Finished, False)

Silver prerequisites validated.


In [3]:
%%sql
CREATE OR REPLACE VIEW LH_Silver.vw_silver_rides_ranked AS
SELECT *,
       ROW_NUMBER() OVER (
         PARTITION BY ride_id
         ORDER BY staging_load_timestamp DESC, bronze_ingestion_timestamp DESC, booking_timestamp DESC
       ) AS row_num
FROM LH_Staging.staging_rides
WHERE ride_id IS NOT NULL AND TRIM(ride_id) <> '';

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql
CREATE OR REPLACE VIEW LH_Silver.vw_silver_rides_clean AS
SELECT
  base_fare, TRY_CAST(booking_timestamp AS TIMESTAMP) booking_timestamp,
  cancellation_reason_id, TRIM(confirmation_number) confirmation_number,
  distance_fare, distance_miles,
  TRIM(driver_id) driver_id, TRIM(driver_license) driver_license,
  TRIM(driver_name) driver_name, TRIM(driver_phone) driver_phone, driver_rating,
  TRIM(dropoff_address) dropoff_address, dropoff_city_id, dropoff_latitude,
  TRIM(dropoff_location_id) dropoff_location_id, dropoff_longitude,
  TRY_CAST(dropoff_timestamp AS TIMESTAMP) dropoff_timestamp, duration_minutes,
  TRIM(license_plate) license_plate,
  TRIM(passenger_email) passenger_email, TRIM(passenger_id) passenger_id,
  TRIM(passenger_name) passenger_name, TRIM(passenger_phone) passenger_phone,
  payment_method_id, TRIM(pickup_address) pickup_address, pickup_city_id, pickup_latitude,
  TRIM(pickup_location_id) pickup_location_id, pickup_longitude,
  TRY_CAST(pickup_timestamp AS TIMESTAMP) pickup_timestamp, rating, TRIM(ride_id) ride_id,
  ride_status_id, subtotal, surge_multiplier, time_fare, tip_amount, total_fare,
  TRIM(vehicle_color) vehicle_color, TRIM(vehicle_id) vehicle_id, vehicle_make_id,
  TRIM(vehicle_model) vehicle_model, vehicle_type_id,
  bronze_ingestion_timestamp, staging_load_timestamp
FROM LH_Silver.vw_silver_rides_ranked
WHERE row_num = 1;

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
%%sql

CREATE TABLE IF NOT EXISTS LH_Silver.driver_scd (
  driver_key BIGINT, driver_id STRING, driver_license STRING, driver_name STRING,
  driver_phone STRING, effective_from TIMESTAMP, effective_to TIMESTAMP, is_current BOOLEAN
) USING DELTA;

CREATE OR REPLACE VIEW LH_Silver.vw_driver_latest AS
SELECT driver_id, driver_license, driver_name, driver_phone, change_timestamp
FROM (
  SELECT driver_id, driver_license, driver_name, driver_phone,
         COALESCE(bronze_ingestion_timestamp, staging_load_timestamp, booking_timestamp) change_timestamp,
         ROW_NUMBER() OVER (
           PARTITION BY driver_id
           ORDER BY bronze_ingestion_timestamp DESC NULLS LAST,
                    staging_load_timestamp DESC NULLS LAST,
                    booking_timestamp DESC NULLS LAST,
                    ride_id DESC
         ) rn
  FROM LH_Silver.vw_silver_rides_clean
  WHERE driver_id IS NOT NULL AND TRIM(driver_id) <> ''
) x WHERE rn = 1;

CREATE OR REPLACE VIEW LH_Silver.vw_driver_changes AS
SELECT s.driver_id,s.driver_license,s.driver_name,s.driver_phone,s.change_timestamp,
       CASE WHEN t.driver_id IS NULL THEN 'NEW' ELSE 'CHANGED' END change_type
FROM LH_Silver.vw_driver_latest s
LEFT JOIN LH_Silver.driver_scd t ON s.driver_id=t.driver_id AND t.is_current=TRUE
WHERE t.driver_id IS NULL
   OR NOT (s.driver_license <=> t.driver_license AND s.driver_name <=> t.driver_name AND s.driver_phone <=> t.driver_phone);

MERGE INTO LH_Silver.driver_scd t
USING (SELECT driver_id,change_timestamp FROM LH_Silver.vw_driver_changes WHERE change_type='CHANGED') s
ON t.driver_id=s.driver_id AND t.is_current=TRUE
WHEN MATCHED THEN UPDATE SET t.effective_to=s.change_timestamp,t.is_current=FALSE;

INSERT INTO LH_Silver.driver_scd
SELECT k.max_key + ROW_NUMBER() OVER (ORDER BY c.driver_id),c.driver_id,c.driver_license,c.driver_name,c.driver_phone,
       c.change_timestamp,CAST(NULL AS TIMESTAMP),TRUE
FROM LH_Silver.vw_driver_changes c
CROSS JOIN (SELECT COALESCE(MAX(driver_key),0) max_key FROM LH_Silver.driver_scd) k;

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 11, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 4 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
%%sql

CREATE TABLE IF NOT EXISTS LH_Silver.passenger_scd (
  passenger_key BIGINT, passenger_id STRING, passenger_name STRING, passenger_phone STRING,
  passenger_email STRING, effective_from TIMESTAMP, effective_to TIMESTAMP, is_current BOOLEAN
) USING DELTA;

CREATE OR REPLACE VIEW LH_Silver.vw_passenger_latest AS
SELECT passenger_id,passenger_name,passenger_phone,passenger_email,change_timestamp
FROM (
  SELECT passenger_id,passenger_name,passenger_phone,passenger_email,
         COALESCE(bronze_ingestion_timestamp, staging_load_timestamp, booking_timestamp) change_timestamp,
         ROW_NUMBER() OVER (
           PARTITION BY passenger_id
           ORDER BY bronze_ingestion_timestamp DESC NULLS LAST,
                    staging_load_timestamp DESC NULLS LAST,
                    booking_timestamp DESC NULLS LAST,
                    ride_id DESC
         ) rn
  FROM LH_Silver.vw_silver_rides_clean
  WHERE passenger_id IS NOT NULL AND TRIM(passenger_id) <> ''
) x WHERE rn=1;

CREATE OR REPLACE VIEW LH_Silver.vw_passenger_changes AS
SELECT s.passenger_id,s.passenger_name,s.passenger_phone,s.passenger_email,s.change_timestamp,
       CASE WHEN t.passenger_id IS NULL THEN 'NEW' ELSE 'CHANGED' END change_type
FROM LH_Silver.vw_passenger_latest s
LEFT JOIN LH_Silver.passenger_scd t ON s.passenger_id=t.passenger_id AND t.is_current=TRUE
WHERE t.passenger_id IS NULL
   OR NOT (s.passenger_name <=> t.passenger_name AND s.passenger_phone <=> t.passenger_phone AND s.passenger_email <=> t.passenger_email);

MERGE INTO LH_Silver.passenger_scd t
USING (SELECT passenger_id,change_timestamp FROM LH_Silver.vw_passenger_changes WHERE change_type='CHANGED') s
ON t.passenger_id=s.passenger_id AND t.is_current=TRUE
WHEN MATCHED THEN UPDATE SET t.effective_to=s.change_timestamp,t.is_current=FALSE;

INSERT INTO LH_Silver.passenger_scd
SELECT k.max_key + ROW_NUMBER() OVER (ORDER BY c.passenger_id),c.passenger_id,c.passenger_name,c.passenger_phone,c.passenger_email,
       c.change_timestamp,CAST(NULL AS TIMESTAMP),TRUE
FROM LH_Silver.vw_passenger_changes c
CROSS JOIN (SELECT COALESCE(MAX(passenger_key),0) max_key FROM LH_Silver.passenger_scd) k;

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 16, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 4 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
%%sql

CREATE TABLE IF NOT EXISTS LH_Silver.city_scd (
  city_key BIGINT, city_id BIGINT, city STRING, region STRING, state STRING,
  effective_from TIMESTAMP, effective_to TIMESTAMP, is_current BOOLEAN
) USING DELTA;
CREATE OR REPLACE VIEW LH_Silver.vw_city_source AS
SELECT city_id,city,region,state FROM (
  SELECT city_id,TRIM(city) city,TRIM(region) region,TRIM(state) state,
         ROW_NUMBER() OVER(PARTITION BY city_id ORDER BY city_id) rn
  FROM LH_Bronze.map_cities WHERE city_id IS NOT NULL
) x WHERE rn=1;
CREATE OR REPLACE VIEW LH_Silver.vw_city_changes AS
SELECT b.city_id,b.city,b.region,b.state,CASE WHEN s.city_id IS NULL THEN 'NEW' ELSE 'CHANGED' END change_type
FROM LH_Silver.vw_city_source b LEFT JOIN LH_Silver.city_scd s ON b.city_id=s.city_id AND s.is_current=TRUE
WHERE s.city_id IS NULL OR NOT(b.city <=> s.city AND b.region <=> s.region AND b.state <=> s.state);
MERGE INTO LH_Silver.city_scd t USING (SELECT * FROM LH_Silver.vw_city_changes WHERE change_type='CHANGED') s
ON t.city_id=s.city_id AND t.is_current=TRUE
WHEN MATCHED THEN UPDATE SET t.effective_to=CURRENT_TIMESTAMP(),t.is_current=FALSE;
INSERT INTO LH_Silver.city_scd
SELECT k.max_key+ROW_NUMBER() OVER(ORDER BY c.city_id),c.city_id,c.city,c.region,c.state,CURRENT_TIMESTAMP(),CAST(NULL AS TIMESTAMP),TRUE
FROM LH_Silver.vw_city_changes c CROSS JOIN (SELECT COALESCE(MAX(city_key),0) max_key FROM LH_Silver.city_scd) k;

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 21, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 4 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
%%sql

CREATE TABLE IF NOT EXISTS LH_Silver.vehicle_type_scd (
  vehicle_type_key BIGINT,vehicle_type_id BIGINT,vehicle_type STRING,description STRING,
  base_rate DOUBLE,per_mile DOUBLE,per_minute DOUBLE,effective_from TIMESTAMP,effective_to TIMESTAMP,is_current BOOLEAN
) USING DELTA;
CREATE OR REPLACE VIEW LH_Silver.vw_vehicle_type_source AS
SELECT vehicle_type_id,vehicle_type,description,base_rate,per_mile,per_minute FROM (
  SELECT vehicle_type_id,TRIM(vehicle_type) vehicle_type,TRIM(description) description,base_rate,per_mile,per_minute,
         ROW_NUMBER() OVER(PARTITION BY vehicle_type_id ORDER BY vehicle_type_id) rn
  FROM LH_Bronze.map_vehicle_types WHERE vehicle_type_id IS NOT NULL
) x WHERE rn=1;
CREATE OR REPLACE VIEW LH_Silver.vw_vehicle_type_changes AS
SELECT b.*,CASE WHEN s.vehicle_type_id IS NULL THEN 'NEW' ELSE 'CHANGED' END change_type
FROM LH_Silver.vw_vehicle_type_source b LEFT JOIN LH_Silver.vehicle_type_scd s
ON b.vehicle_type_id=s.vehicle_type_id AND s.is_current=TRUE
WHERE s.vehicle_type_id IS NULL OR NOT(b.vehicle_type <=> s.vehicle_type AND b.description <=> s.description
  AND b.base_rate <=> s.base_rate AND b.per_mile <=> s.per_mile AND b.per_minute <=> s.per_minute);
MERGE INTO LH_Silver.vehicle_type_scd t USING (SELECT * FROM LH_Silver.vw_vehicle_type_changes WHERE change_type='CHANGED') s
ON t.vehicle_type_id=s.vehicle_type_id AND t.is_current=TRUE
WHEN MATCHED THEN UPDATE SET t.effective_to=CURRENT_TIMESTAMP(),t.is_current=FALSE;
INSERT INTO LH_Silver.vehicle_type_scd
SELECT k.max_key+ROW_NUMBER() OVER(ORDER BY c.vehicle_type_id),c.vehicle_type_id,c.vehicle_type,c.description,c.base_rate,c.per_mile,c.per_minute,
       CURRENT_TIMESTAMP(),CAST(NULL AS TIMESTAMP),TRUE
FROM LH_Silver.vw_vehicle_type_changes c CROSS JOIN (SELECT COALESCE(MAX(vehicle_type_key),0) max_key FROM LH_Silver.vehicle_type_scd) k;

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 26, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 4 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
%%sql

CREATE TABLE IF NOT EXISTS LH_Silver.map_vehicle_makes USING DELTA AS
SELECT CAST(NULL AS BIGINT) vehicle_make_id,CAST(NULL AS STRING) vehicle_make WHERE 1=0;
MERGE INTO LH_Silver.map_vehicle_makes t USING (
SELECT vehicle_make_id,vehicle_make FROM (SELECT vehicle_make_id,TRIM(vehicle_make) vehicle_make,
ROW_NUMBER() OVER(PARTITION BY vehicle_make_id ORDER BY vehicle_make) rn FROM LH_Bronze.map_vehicle_makes
WHERE vehicle_make_id IS NOT NULL) x WHERE rn=1) s ON t.vehicle_make_id=s.vehicle_make_id
WHEN MATCHED AND NOT(t.vehicle_make <=> s.vehicle_make) THEN UPDATE SET t.vehicle_make=s.vehicle_make
WHEN NOT MATCHED THEN INSERT *;

CREATE TABLE IF NOT EXISTS LH_Silver.map_payment_methods USING DELTA AS
SELECT CAST(NULL AS BIGINT) payment_method_id,CAST(NULL AS STRING) payment_method,CAST(NULL AS BOOLEAN) is_card,CAST(NULL AS BOOLEAN) requires_auth WHERE 1=0;
MERGE INTO LH_Silver.map_payment_methods t USING (
SELECT payment_method_id,payment_method,is_card,requires_auth FROM (SELECT payment_method_id,TRIM(payment_method) payment_method,is_card,requires_auth,
ROW_NUMBER() OVER(PARTITION BY payment_method_id ORDER BY payment_method) rn FROM LH_Bronze.map_payment_methods WHERE payment_method_id IS NOT NULL) x WHERE rn=1) s
ON t.payment_method_id=s.payment_method_id
WHEN MATCHED AND NOT(t.payment_method <=> s.payment_method AND t.is_card <=> s.is_card AND t.requires_auth <=> s.requires_auth)
THEN UPDATE SET t.payment_method=s.payment_method,t.is_card=s.is_card,t.requires_auth=s.requires_auth
WHEN NOT MATCHED THEN INSERT *;

CREATE TABLE IF NOT EXISTS LH_Silver.map_ride_statuses USING DELTA AS
SELECT CAST(NULL AS BIGINT) ride_status_id,CAST(NULL AS STRING) ride_status,CAST(NULL AS BOOLEAN) is_completed WHERE 1=0;
MERGE INTO LH_Silver.map_ride_statuses t USING (
SELECT ride_status_id,ride_status,is_completed FROM (SELECT ride_status_id,TRIM(ride_status) ride_status,is_completed,
ROW_NUMBER() OVER(PARTITION BY ride_status_id ORDER BY ride_status) rn FROM LH_Bronze.map_ride_statuses WHERE ride_status_id IS NOT NULL) x WHERE rn=1) s
ON t.ride_status_id=s.ride_status_id
WHEN MATCHED AND NOT(t.ride_status <=> s.ride_status AND t.is_completed <=> s.is_completed)
THEN UPDATE SET t.ride_status=s.ride_status,t.is_completed=s.is_completed
WHEN NOT MATCHED THEN INSERT *;

CREATE TABLE IF NOT EXISTS LH_Silver.map_cancellation_reasons USING DELTA AS
SELECT CAST(NULL AS BIGINT) cancellation_reason_id,CAST(NULL AS STRING) cancellation_reason WHERE 1=0;
MERGE INTO LH_Silver.map_cancellation_reasons t USING (
SELECT cancellation_reason_id,cancellation_reason FROM (SELECT cancellation_reason_id,TRIM(cancellation_reason) cancellation_reason,
ROW_NUMBER() OVER(PARTITION BY cancellation_reason_id ORDER BY cancellation_reason) rn FROM LH_Bronze.map_cancellation_reasons WHERE cancellation_reason_id IS NOT NULL) x WHERE rn=1) s
ON t.cancellation_reason_id=s.cancellation_reason_id
WHEN MATCHED AND NOT(t.cancellation_reason <=> s.cancellation_reason) THEN UPDATE SET t.cancellation_reason=s.cancellation_reason
WHEN NOT MATCHED THEN INSERT *;

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 34, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 4 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 4 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 4 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 4 fields>

In [10]:
%%sql

CREATE OR REPLACE VIEW LH_Silver.vw_rides_obt AS
SELECT r.ride_id,r.confirmation_number,r.booking_timestamp,
  p.passenger_key,d.driver_key,pc.city_key pickup_city_key,dc.city_key dropoff_city_key,vt.vehicle_type_key,
  r.passenger_id,r.driver_id,r.vehicle_id,r.vehicle_type_id,r.vehicle_make_id,r.pickup_city_id,r.dropoff_city_id,
  r.payment_method_id,r.ride_status_id,r.cancellation_reason_id,r.vehicle_model,r.vehicle_color,r.license_plate,vm.vehicle_make,
  vt.vehicle_type,vt.description vehicle_type_description,vt.base_rate vehicle_type_base_rate,vt.per_mile vehicle_type_per_mile,vt.per_minute vehicle_type_per_minute,
  p.passenger_name,p.passenger_phone,p.passenger_email,d.driver_name,d.driver_phone,d.driver_license,r.driver_rating,
  r.pickup_location_id,r.pickup_address,r.pickup_latitude,r.pickup_longitude,r.pickup_timestamp,pc.city pickup_city,pc.region pickup_region,pc.state pickup_state,
  r.dropoff_location_id,r.dropoff_address,r.dropoff_latitude,r.dropoff_longitude,r.dropoff_timestamp,dc.city dropoff_city,dc.region dropoff_region,dc.state dropoff_state,
  rs.ride_status,rs.is_completed,r.distance_miles,r.duration_minutes,r.rating,r.base_fare,r.distance_fare,r.time_fare,r.surge_multiplier,r.subtotal,r.tip_amount,r.total_fare,
  pm.payment_method,pm.is_card,pm.requires_auth,cr.cancellation_reason,r.bronze_ingestion_timestamp,r.staging_load_timestamp
FROM LH_Silver.vw_silver_rides_clean r
LEFT JOIN LH_Silver.passenger_scd p ON r.passenger_id=p.passenger_id
  AND COALESCE(r.bronze_ingestion_timestamp,r.staging_load_timestamp,r.booking_timestamp)>=p.effective_from
  AND (COALESCE(r.bronze_ingestion_timestamp,r.staging_load_timestamp,r.booking_timestamp)<p.effective_to OR p.effective_to IS NULL)
LEFT JOIN LH_Silver.driver_scd d ON r.driver_id=d.driver_id
  AND COALESCE(r.bronze_ingestion_timestamp,r.staging_load_timestamp,r.booking_timestamp)>=d.effective_from
  AND (COALESCE(r.bronze_ingestion_timestamp,r.staging_load_timestamp,r.booking_timestamp)<d.effective_to OR d.effective_to IS NULL)
LEFT JOIN LH_Silver.city_scd pc ON r.pickup_city_id=pc.city_id AND pc.is_current=TRUE
LEFT JOIN LH_Silver.city_scd dc ON r.dropoff_city_id=dc.city_id AND dc.is_current=TRUE
LEFT JOIN LH_Silver.vehicle_type_scd vt ON r.vehicle_type_id=vt.vehicle_type_id AND vt.is_current=TRUE
LEFT JOIN LH_Silver.map_vehicle_makes vm ON r.vehicle_make_id=vm.vehicle_make_id
LEFT JOIN LH_Silver.map_payment_methods pm ON r.payment_method_id=pm.payment_method_id
LEFT JOIN LH_Silver.map_ride_statuses rs ON r.ride_status_id=rs.ride_status_id
LEFT JOIN LH_Silver.map_cancellation_reasons cr ON r.cancellation_reason_id=cr.cancellation_reason_id;

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 35, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [11]:
%%sql
CREATE TABLE IF NOT EXISTS LH_Silver.rides_obt USING DELTA AS SELECT * FROM LH_Silver.vw_rides_obt WHERE 1=0;
MERGE INTO LH_Silver.rides_obt t USING LH_Silver.vw_rides_obt s ON t.ride_id=s.ride_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 37, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 1 rows and 4 fields>

In [12]:
%%sql

SELECT 'driver_scd' table_name,driver_id business_key,COUNT(*) current_count
FROM LH_Silver.driver_scd WHERE is_current=TRUE GROUP BY driver_id HAVING COUNT(*)<>1
UNION ALL
SELECT 'passenger_scd',passenger_id,COUNT(*) FROM LH_Silver.passenger_scd
WHERE is_current=TRUE GROUP BY passenger_id HAVING COUNT(*)<>1;

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 38, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 3 fields>

In [13]:
%%sql
SELECT COUNT(*) total_rows,COUNT(DISTINCT ride_id) unique_rides,
       COUNT(*)-COUNT(DISTINCT ride_id) duplicate_rides
FROM LH_Silver.rides_obt;

StatementMeta(, 9ec902a4-9fef-40a5-a733-7ce18f3201a5, 39, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 3 fields>